# Практикум 13. Автоматическое реферирование: экстрактивные (TextRank) и абстрактивные методы

В лаборатории 13 вы строили граф сходства предложений, смотрели, как итерации PageRank «раздувают» важные вершины, и сравнивали экстрактивный реферат с абстрактивным. Здесь сделаем то же самое в коде:

1. разобьём текст о Самарканде на предложения и подготовим «мешки» основ слов;
2. построим граф сходства предложений и посчитаем PageRank степенным методом с нуля, затем сверимся с `networkx.pagerank`;
3. соберём экстрактивный реферат и оценим его метрикой ROUGE-1, посчитанной вручную;
4. применим TextRank к словам, чтобы найти ключевые слова;
5. сравним экстрактивный реферат с абстрактивным и разберём идею pointer-generator (копировать или порождать).

*Практикум к лаборатории 13 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Текст и разбиение на предложения

Экстрактивный реферат собирается из целых предложений исходного текста, поэтому сначала текст нужно правильно разрезать. Простое правило: предложение заканчивается на `.`, `!`, `?`, после которых идёт пробел и заглавная буква или цифра. Ниже наш текст (13 предложений) и пример, где это правило ломается: сокращения «г.» и инициалы.

In [ ]:
text = """Самарканд — один из древнейших городов Центральной Азии, ему более 2700 лет. Древние греки называли город Мараканда. Самарканд стоял на Великом шёлковом пути, и через город шли караваны из Китая в Персию и дальше на запад. В XIV веке Амир Темур сделал Самарканд столицей своего государства. При Темуре в Самарканде построили мечеть Биби-Ханым и много других зданий. Внук Темура Улугбек был не только правителем, но и учёным-астрономом. В 1420-х годах Улугбек построил в Самарканде обсерваторию. Астрономы обсерватории составили каталог, в котором больше тысячи звёзд. Площадь Регистан окружают три медресе: медресе Улугбека, Шердор и Тилля-Кари. Медресе Улугбека — самое старое здание на Регистане, его построили в XV веке. Амир Темур похоронен в мавзолее Гур-Эмир. В 2001 году исторический центр Самарканда включили в список Всемирного наследия ЮНЕСКО. Каждый год тысячи туристов приезжают в Самарканд, чтобы увидеть Регистан, Гур-Эмир и Шахи-Зинду."""

import re
import numpy as np

def split_sentences(text):
    # режем после . ! ? …, если дальше пробел и заглавная буква, цифра или кавычка
    return [s.strip() for s in re.split(r"(?<=[.!?…])\s+(?=[«\"A-ZА-ЯЁ0-9])", text.strip()) if s.strip()]

sents = split_sentences(text)
for i, s in enumerate(sents):
    print(f"{i:>2}. {s}")
print("\nвсего предложений:", len(sents))

print("\nловушка: сокращение «г.» и инициалы:")
for s in split_sentences("В 1417 г. Улугбек начал строить медресе. В. В. Бартольд написал книгу «Улугбек и его время»."):
    print("   |", s)

## 2. Подготовка: основы слов без стоп-слов

Сходство предложений будем считать по общим словам. Чтобы «Самарканд», «Самарканда» и «Самарканде» считались одним словом, приведём слова к основам стеммером Snowball из `nltk`. Служебные слова (предлоги, союзы, местоимения) уберём списком стоп-слов: они есть почти в каждом предложении и не несут темы.

In [ ]:
def tokens(text):
    return re.findall(r"[а-яёa-z][а-яёa-zʻʼ'-]*", text.lower())

from nltk.stem.snowball import SnowballStemmer
stem = SnowballStemmer("russian").stem

STOP = set("""в во на и с со по из к у о об от до за при для а но или не ни же ли бы то что это как так
    только его её ее их ему ей им он она оно они был была было были который которая котором которые
    свой своего своей один более больше много других чтобы каждый""".split())

def content_words(sentence):
    # основы слов без стоп-слов и однобуквенных обрывков («1420-х» -> «х»)
    return [stem(w) for w in tokens(sentence) if w not in STOP and len(w) > 1]

bags = [content_words(s) for s in sents]
for i, b in enumerate(bags[:5]):
    print(f"{i:>2}.", " ".join(b))
print("...")

## 3. Граф сходства предложений

Вершины графа — предложения, вес ребра — сходство. В оригинальной статье TextRank (Mihalcea, Tarau, 2004) сходство двух предложений — число общих слов, делённое на сумму логарифмов их длин:

$$w_{ij} = \frac{|\{w : w \in S_i \text{ и } w \in S_j\}|}{\log |S_i| + \log |S_j|}$$

Логарифм в знаменателе не даёт длинным предложениям выигрывать только за счёт длины.

In [ ]:
import math
import pandas as pd

def overlap_similarity(a, b):
    # сходство из статьи TextRank: общие слова / (log|S_i| + log|S_j|)
    common = len(set(a) & set(b))
    if common == 0 or len(a) < 2 or len(b) < 2:
        return 0.0
    return common / (math.log(len(a)) + math.log(len(b)))

n = len(sents)
W = np.array([[overlap_similarity(bags[i], bags[j]) if i != j else 0.0 for j in range(n)] for i in range(n)])
print(pd.DataFrame(W).round(2).to_string())
print(f"\nрёбер с весом > 0: {int((W > 0).sum() / 2)} из {n * (n - 1) // 2}")
i, j = 4, 6
print(f"пример, предложения {i}–{j}: {sorted(set(bags[i]) & set(bags[j]))}, "
      f"|S{i}| = {len(bags[i])}, |S{j}| = {len(bags[j])}, w = {W[i, j]:.3f}")

## 4. PageRank степенным методом

Важное предложение похоже на другие важные предложения. PageRank превращает эту идею в формулу: «случайный читатель» переходит от предложения к предложению по рёбрам (с вероятностью, пропорциональной весу), а с вероятностью $1-d$ прыгает в случайное предложение:

$$PR(i) = \frac{1-d}{N} + d \sum_{j} \frac{w_{ji}}{\sum_k w_{jk}}\, PR(j), \qquad d = 0.85$$

Начинаем с $PR = 1/N$ и повторяем формулу, пока значения не перестанут меняться (степенной метод). Ниже печатаются первые итерации и последняя.


Сверимся с `networkx.pagerank`. В статье TextRank формула записана без деления на $N$ ($1-d$ вместо $\frac{1-d}{N}$); это только масштаб, порядок предложений тот же.

In [ ]:
def pagerank(W, d=0.85, tol=1e-10, max_iter=100, verbose=False):
    n = len(W)
    out = W.sum(axis=1)
    M = np.divide(W, out[:, None], out=np.zeros_like(W), where=out[:, None] > 0).T   # M[i, j] = w_ji / sum_k w_jk: доля веса, которую j отдаёт i
    dangling = out == 0                                                              # вершины без рёбер раздают вес всем поровну (как в networkx)
    p = np.full(n, 1 / n)
    history = [p]
    for it in range(1, max_iter + 1):
        p_new = (1 - d) / n + d * (M @ p + p[dangling].sum() / n)
        delta = np.abs(p_new - p).sum()
        p = p_new
        history.append(p)
        if verbose and (it <= 3 or delta < tol):
            print(f"итерация {it:>2}: delta = {delta:.2e}   " + " ".join(f"{x:.3f}" for x in p))
        if delta < tol:
            break
    return p, history

scores, history = pagerank(W, verbose=True)

import networkx as nx

G = nx.from_numpy_array(W)                          # неориентированный взвешенный граф из матрицы W
pr = nx.pagerank(G, alpha=0.85, weight="weight", tol=1e-12)
nx_scores = np.array([pr[i] for i in range(n)])
print(f"max |наш PageRank - networkx| = {np.abs(scores - nx_scores).max():.2e}")

order = np.argsort(-scores)
print("\nпредложения по убыванию PageRank:")
for r, i in enumerate(order, start=1):
    print(f"{r:>2}. [{i:>2}] {scores[i]:.3f}  {sents[i][:70]}")

Слева — граф: размер вершины пропорционален PageRank, толщина ребра — весу, оранжевым отмечены три лучших предложения. Справа — как меняется PageRank каждого предложения по итерациям. Метод сходится за несколько десятков шагов.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.4, 1]})
pos = nx.circular_layout(G)
widths = [3 * G[u][v]["weight"] for u, v in G.edges()]
top = set(order[:3])
nx.draw_networkx_edges(G, pos, ax=ax[0], width=widths, alpha=0.35)
nx.draw_networkx_nodes(G, pos, ax=ax[0], node_size=[12000 * s for s in scores],
                       node_color=["#D9480F" if i in top else "#2B55C9" for i in range(n)])
nx.draw_networkx_labels(G, pos, ax=ax[0], font_color="white", font_size=9)
ax[0].set_title("Граф предложений (размер = PageRank)")
ax[0].axis("off")

H = np.array(history)
for i in range(n):
    ax[1].plot(H[:, i], color="#D9480F" if i in top else "#9AA5B1", lw=2 if i in top else 1)
ax[1].set_xlabel("итерация")
ax[1].set_ylabel("PageRank")
ax[1].set_title("Сходимость PageRank")
plt.tight_layout()
plt.show()

## 5. Экстрактивный реферат и ROUGE-1

Реферат — $k$ предложений с наибольшим PageRank, **в порядке исходного текста** (иначе текст будет нечитаемым). Степень сжатия задаём долей предложений `ratio`, как ползунком в лаборатории.


Второй вопрос — важен ли выбор меры сходства? Заменим формулу TextRank косинусом TF-IDF (из практикума 11) и сравним, какие предложения попадут в реферат.

In [ ]:
def summarize(sents, scores, ratio=0.25):
    # k лучших предложений, но в порядке исходного текста
    k = max(1, round(ratio * len(sents)))
    chosen = sorted(int(i) for i in np.argsort(-scores)[:k])
    return [sents[i] for i in chosen], chosen

summary, chosen = summarize(sents, scores, ratio=0.25)
print(f"выбраны предложения: {chosen}\n")
for i, s in enumerate(sents):
    mark = ">>" if i in chosen else "  "
    print(f"{mark} {i:>2}. {s}")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(analyzer=lambda s: s)          # векторизуем уже готовые списки основ
C = cosine_similarity(vec.fit_transform(bags))
np.fill_diagonal(C, 0)
cos_scores, _ = pagerank(C)
print("тройка по формуле TextRank:", sorted(int(i) for i in np.argsort(-scores)[:3]))
print("тройка по косинусу TF-IDF:    ", sorted(int(i) for i in np.argsort(-cos_scores)[:3]))
print(f"корреляция двух наборов оценок PageRank: {np.corrcoef(scores, cos_scores)[0, 1]:.2f}")

Как оценить реферат? Нужен эталон, написанный человеком. Метрика **ROUGE-1** считает совпадающие слова (униграммы) кандидата и эталона: полнота $R$ = совпадения / длина эталона, точность $P$ = совпадения / длина кандидата, $F_1$ — их гармоническое среднее. Подробно ROUGE и BLEU разберём в практикуме 18.

Сравним TextRank с двумя простыми базовыми методами: **Lead-k** (первые $k$ предложений, для новостей это сильный бейзлайн) и случайными $k$ предложениями. Посчитаем ROUGE-1 по словоформам и по основам.

In [ ]:
# эталонный реферат, написанный человеком (абстрактивный: своими словами)
reference = ("Самарканд — древний город на Великом шёлковом пути, ему больше 2700 лет. "
             "Амир Темур сделал его своей столицей, а его внук Улугбек построил здесь обсерваторию. "
             "Исторический центр города с площадью Регистан входит в список Всемирного наследия ЮНЕСКО.")

from collections import Counter

def rouge1(candidate, reference, prep):
    c, r = Counter(prep(candidate)), Counter(prep(reference))
    overlap = sum((c & r).values())                # каждое слово засчитываем не больше раз, чем оно есть в эталоне
    p = overlap / max(1, sum(c.values()))
    rec = overlap / max(1, sum(r.values()))
    f = 2 * p * rec / (p + rec) if p + rec else 0.0
    return p, rec, f

k = len(chosen)
rng = np.random.default_rng(0)
candidates = {
    "TextRank": " ".join(summary),
    f"Lead-{k}": " ".join(sents[:k]),
    "случайные": " ".join(sents[i] for i in sorted(rng.choice(n, size=k, replace=False))),
}
print(f"{'':<12}{'словоформы':>22}{'основы':>26}")
print(f"{'':<12}{'P':>8}{'R':>7}{'F1':>7}{'P':>9}{'R':>7}{'F1':>7}")
for name, cand in candidates.items():
    a = rouge1(cand, reference, tokens)
    b = rouge1(cand, reference, lambda t: [stem(w) for w in tokens(t)])
    print(f"{name:<12}" + "".join(f"{x:7.2f}" for x in a) + "  " + "".join(f"{x:7.2f}" for x in b))

На таком маленьком тексте разница между методами невелика, и одно число не стоит переоценивать: эталон всего один, а ROUGE не видит смысла, только совпадение слов. Обратите внимание, что сравнение по основам повышает оценки: начинают совпадать «своей» и «своего», «древний» и «древнейших», «города» и «городов».

## 6. Ключевые слова: TextRank на словах

Тот же алгоритм работает и для слов. Вершины — основы слов, ребро соединяет слова, стоящие рядом (в окне `window`) после удаления стоп-слов. PageRank выбирает слова, у которых много «соседей». В оригинальном TextRank в граф берут только существительные и прилагательные; для русского это можно сделать с `pymorphy3` (см. практикум 3), здесь мы обходимся стоп-словами.

In [ ]:
def keywords(sents, window=2, top=10):
    G = nx.Graph()
    surface = {}
    for s in sents:
        words = [w for w in tokens(s) if w not in STOP and len(w) > 2]
        stems = [stem(w) for w in words]
        for w, st in zip(words, stems):
            surface.setdefault(st, Counter())[w] += 1
        for i in range(len(stems)):                       # соединяем слова, стоящие ближе window
            for j in range(i + 1, min(i + window, len(stems))):
                if stems[i] != stems[j]:
                    G.add_edge(stems[i], stems[j], weight=G.get_edge_data(stems[i], stems[j], {"weight": 0})["weight"] + 1)
    pr = nx.pagerank(G, weight="weight")
    best = sorted(pr, key=pr.get, reverse=True)[:top]
    return [(surface[st].most_common(1)[0][0], pr[st]) for st in best], G

kw, KG = keywords(sents, window=2)
print(f"граф слов: {KG.number_of_nodes()} вершин, {KG.number_of_edges()} рёбер")
for w, s in kw:
    print(f"   {w:<16} {s:.3f}")

## 7. Экстрактивный и абстрактивный реферат

**Экстрактивный** метод копирует предложения целиком, **абстрактивный** пишет новый текст своими словами: объединяет факты, сокращает, перефразирует. Наш эталон абстрактивный. Посмотрим, какие его слова встречаются в исходном тексте (скопированы), а какие новые (в квадратных скобках — «порождены»), и какая доля биграмм реферата отсутствует в источнике.

In [ ]:
source_stems = {stem(w) for w in tokens(text)}
abs_words = re.findall(r"\S+", reference)

copied = 0
line = []
for w in abs_words:
    toks = tokens(w)
    if not toks:                                   # тире и числа не считаем
        line.append(w)
        continue
    if all(stem(t) in source_stems for t in toks):
        line.append(w)
        copied += 1
    else:
        line.append("[" + w + "]")                 # слова нет в источнике: «порождено»
print(" ".join(line))
total = sum(1 for w in abs_words if tokens(w))
print(f"\nскопировано из источника слов: {copied}/{total} = {copied / total:.0%}")

def bigrams(t):
    # пары соседних слов внутри каждого предложения
    return {b for s in split_sentences(t) for b in zip(tokens(s), tokens(s)[1:])}

src_bi = bigrams(text)
for name, cand in [("экстрактивный", " ".join(summary)), ("абстрактивный", reference)]:
    b = bigrams(cand)
    print(f"новых биграмм ({name}): {len(b - src_bi)}/{len(b)} = {len(b - src_bi) / len(b):.0%}")

Нейросетевые абстрактивные модели часто устроены как **pointer-generator** (See et al., 2017): на каждом шаге модель решает, породить слово из словаря или скопировать его из источника. Итоговое распределение смешивает два: $P(w) = p_{gen}\, P_{vocab}(w) + (1 - p_{gen}) \sum_{i: x_i = w} a_i$, где $a_i$ — внимание (attention) на $i$-е слово источника. Числа ниже придуманы для иллюстрации: модель выбирает слово после «Амир Темур сделал Самарканд ...». Сам генератор предпочёл бы «центром», но копирование из источника перевешивает. Попробуйте поменять `p_gen` (например, 0.9): при каком значении побеждает «своё» слово генератора?

In [ ]:
# pointer-generator: смешиваем «породить из словаря» и «скопировать из источника»
p_gen = 0.3
p_vocab = {"центром": 0.40, "столицей": 0.25, "городом": 0.20, "своей": 0.15}
p_attn = {"столицей": 0.70, "государства": 0.20, "Самарканд": 0.10}
words = sorted(set(p_vocab) | set(p_attn))
P = {w: p_gen * p_vocab.get(w, 0) + (1 - p_gen) * p_attn.get(w, 0) for w in words}
for w in sorted(P, key=P.get, reverse=True):
    print(f"   {w:<14} {p_gen:.1f}·{p_vocab.get(w, 0):.2f} + {1 - p_gen:.1f}·{p_attn.get(w, 0):.2f} = {P[w]:.3f}")

### Как это выглядит на настоящей модели

`csebuetnlp/mT5_multilingual_XLSum` — модель mT5-base, дообученная на новостном корпусе XL-Sum (статьи BBC более чем на 40 языках, среди них русский и узбекский). **Внимание: файл модели ≈2.3 ГБ**, загрузка займёт несколько минут; в Colab памяти хватает. Модель обучена писать заголовок-«лид» новости в одно предложение, поэтому реферат будет коротким. Проверьте каждый факт в нём по исходному тексту: абстрактивные модели иногда «галлюцинируют», то есть пишут то, чего в источнике нет. Ячейки работают только в Colab.

In [ ]:
!pip install -q sentencepiece

# абстрактивный реферат настоящей моделью (≈2.3 ГБ); параметры генерации как в карточке модели
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    name = "csebuetnlp/mT5_multilingual_XLSum"
    tok = AutoTokenizer.from_pretrained(name)
    mt5 = AutoModelForSeq2SeqLM.from_pretrained(name)

    article = re.sub(r"\s+", " ", text.strip())
    inputs = tok([article], return_tensors="pt", truncation=True, max_length=512)
    output_ids = mt5.generate(**inputs, max_length=84, no_repeat_ngram_size=2, num_beams=4)[0]
    generated = tok.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    print("mT5:", generated)
    print("ROUGE-1 (P, R, F1) против эталона:", tuple(round(x, 2) for x in rouge1(generated, reference, tokens)))
except Exception as e:
    print("Не удалось загрузить модель (нужен интернет, пакеты transformers и sentencepiece):", e)

## 8. Задания

1. **Сокращения.** Улучшите `split_sentences`, чтобы она не резала текст после «г.» и инициалов (одна заглавная буква с точкой). Проверьте на строке из раздела 1: должно получиться 2 предложения.
2. **Порог рёбер.** Удалите из графа рёбра с весом меньше порога `t` (`W * (W >= t)`) и пересчитайте PageRank для `t = 0, 0.25, 0.3, 0.45`. Как меняется тройка лучших предложений? Что происходит с вершинами, у которых не осталось рёбер?
3. **Коэффициент d.** Посчитайте PageRank для `d = 0.5, 0.85, 0.99`. Сколько итераций нужно до сходимости? Как меняется разброс оценок (max − min)?
4. **Сжатие.** Постройте график ROUGE-1 $F_1$ (по основам) для TextRank и Lead-k при `ratio` от 0.1 до 0.6. При каком сжатии TextRank выигрывает сильнее всего?
5. **Свой текст.** Напишите или вставьте текст из 10–15 предложений на другую тему (например, о своём городе) и получите для него реферат из 3 предложений и 8 ключевых слов. Совет: можно загрузить файл `my_text.txt` через панель «Файлы» и прочитать его `open("my_text.txt", encoding="utf-8").read()`.

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь